# Testing file 
### where we evaluate Zhang's models using the test set

## Preliminaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset


from util.load_data import load_data
from util.evaluation import *
from models.zhang.models import FairLogisticRegression
from models.zhang.learning import train_loop as zhang_train

/Users/lffpl/Projects/falsb/env/falsb/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
batch_size = 64
epochs = 100
lr = 0.001

In [3]:
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'arrhythmia-age'

In [5]:
x, y, a = load_data(data_name)
raw_data = (x, y, a)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
adim = a.shape[1]
zdim = 8

In [7]:
print(xdim, ydim, adim, zdim)

278 1 1 8


In [8]:
print(len(x))

420


## Result file

In [9]:
header = "model_name", "cv_seed", "clas_acc", "dp", "deqodds", "deqopp", "trade_dp", "trade_deqodds", "trade_deqopp", "TN_a0", "FP_a0", "FN_a0", "TP_a0", "TN_a1", "FP_a1", "FN_a1", "TP_a1"
results = []

## Testing loop
#### Each model is evalueted 5 times
#### In the end of each iteration we save the result

### Zhang for DP

In [10]:
fairdef = 'DemPar'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)

    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4DP', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


2026-01-05 15:26:48.369722: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-01-05 15:26:48.530869: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 1 | 0.7591468095779419 | 0.8117194771766663 | 0.5625 | 0.5703125
> 2 | 0.721703052520752 | 0.8106709718704224 | 0.5625 | 0.5703125
> 3 | 0.7108426690101624 | 0.8099626898765564 | 0.5625 | 0.5703125
> 4 | 0.7054281830787659 | 0.8092418313026428 | 0.5625 | 0.5703125


2026-01-05 15:26:48.882892: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 5 | 0.7011069059371948 | 0.8085284233093262 | 0.5625 | 0.5703125
> 6 | 0.6971829533576965 | 0.8078258037567139 | 0.5625 | 0.5703125
> 7 | 0.6935197710990906 | 0.8071325421333313 | 0.5625 | 0.5703125
> 8 | 0.6902415156364441 | 0.8064388036727905 | 0.5625 | 0.5703125


...

> 64 | 0.580752968788147 | 0.7755600810050964 | 0.73046875 | 0.5703125
> 65 | 0.5792920589447021 | 0.7750526666641235 | 0.7421875 | 0.5703125
> 66 | 0.5778417587280273 | 0.774546205997467 | 0.7421875 | 0.5703125
> 67 | 0.5764021873474121 | 0.7740405797958374 | 0.7421875 | 0.5703125
> 68 | 0.5749868154525757 | 0.7735248804092407 | 0.75 | 0.5703125
> 69 | 0.5735809206962585 | 0.7730097770690918 | 0.75 | 0.5703125
> 70 | 0.5721849203109741 | 0.7724952697753906 | 0.75 | 0.5703125
> 71 | 0.5707986354827881 | 0.7719815969467163 | 0.75 | 0.5703125
> 72 | 0.5694223642349243 | 0.7714685201644897 | 0.75 | 0.5703125
> 73 | 0.5680559277534485 | 0.77095627784729 | 0.75 | 0.5703125
> 74 | 0.5666996836662292 | 0.7704450488090515 | 0.75 | 0.5703125
> 75 | 0.5653661489486694 | 0.7699483633041382 | 0.75 | 0.5703125
> 76 | 0.5640422105789185 | 0.7694523930549622 | 0.75 | 0.5703125
> 77 | 0.5627278685569763 | 0.768957257270813 | 0.75 | 0.5703125
> 78 | 0.5614231824874878 | 0.7684628963470459 | 0.75 | 0.57

2026-01-05 15:27:11.326726: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 27 | 0.605015754699707 | 0.7026559114456177 | 0.73046875 | 0.58984375
> 28 | 0.6023927927017212 | 0.7020878195762634 | 0.7421875 | 0.58984375
> 29 | 0.5997583270072937 | 0.7015430331230164 | 0.74609375 | 0.58984375
> 30 | 0.5972089171409607 | 0.7009679079055786 | 0.74609375 | 0.58984375
> 31 | 0.5946378707885742 | 0.7004117965698242 | 0.75390625 | 0.58984375
> 32 | 0.5921076536178589 | 0.6998655200004578 | 0.76171875 | 0.58984375
> 33 | 0.5896131992340088 | 0.6993165612220764 | 0.76171875 | 0.58984375
> 34 | 0.5871518850326538 | 0.6987695693969727 | 0.7734375 | 0.58984375
> 35 | 0.5847163796424866 | 0.6982243657112122 | 0.7734375 | 0.58984375
> 36 | 0.5823242664337158 | 0.6976761221885681 | 0.7734375 | 0.58984375
> 37 | 0.5799573659896851 | 0.6971291303634644 | 0.7734375 | 0.58984375
> 38 | 0.5776278376579285 | 0.6965823173522949 | 0.77734375 | 0.58984375
> 39 | 0.5753143429756165 | 0.6960352659225464 | 0.78125 | 0.58984375
> 40 | 0.573024570941925 | 0.6954898238182068 | 0.78515625 |

2026-01-05 15:27:34.620428: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 55 | 0.5508152842521667 | 0.6987576484680176 | 0.76171875 | 0.56640625
> 56 | 0.5492886304855347 | 0.6982638835906982 | 0.765625 | 0.56640625
> 57 | 0.5477648973464966 | 0.6977652311325073 | 0.765625 | 0.56640625
> 58 | 0.5462685227394104 | 0.697275698184967 | 0.76953125 | 0.56640625
> 59 | 0.544772744178772 | 0.6967732906341553 | 0.7734375 | 0.56640625
> 60 | 0.5433034896850586 | 0.6962880492210388 | 0.7734375 | 0.56640625
> 61 | 0.5418451428413391 | 0.695795476436615 | 0.7734375 | 0.56640625
> 62 | 0.5403910279273987 | 0.6952964663505554 | 0.7734375 | 0.56640625
> 63 | 0.5389513969421387 | 0.6948032379150391 | 0.7734375 | 0.56640625
> 64 | 0.5375255942344666 | 0.6943129301071167 | 0.77734375 | 0.56640625
> 65 | 0.5361332297325134 | 0.6938302516937256 | 0.77734375 | 0.56640625
> 66 | 0.534756064414978 | 0.6933410167694092 | 0.77734375 | 0.56640625
> 67 | 0.5333902835845947 | 0.6928529739379883 | 0.77734375 | 0.56640625
> 68 | 0.532072901725769 | 0.6923844218254089 | 0.78125 | 0.5664

### Zhang for Eq Odds

In [11]:
fairdef = 'EqOdds'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7591468095779419 | 0.81098473072052 | 0.5625 | 0.5703125
> 2 | 0.7217442989349365 | 0.8093845844268799 | 0.5625 | 0.5703125
> 3 | 0.7108627557754517 | 0.8081002235412598 | 0.5625 | 0.5703125
> 4 | 0.7054427862167358 | 0.806792140007019 | 0.5625 | 0.5703125
> 5 | 0.7011168599128723 | 0.8054993152618408 | 0.5625 | 0.5703125
> 6 | 0.6971899271011353 | 0.8042290806770325 | 0.5625 | 0.5703125
> 7 | 0.6934483051300049 | 0.8029820919036865 | 0.5625 | 0.5703125


2026-01-05 15:28:20.147964: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 8 | 0.6901702284812927 | 0.8017319440841675 | 0.5625 | 0.5703125
> 9 | 0.6869308948516846 | 0.8005017042160034 | 0.5625 | 0.5703125
> 10 | 0.6837291717529297 | 0.7992913722991943 | 0.56640625 | 0.5703125
> 11 | 0.6806378364562988 | 0.7980957627296448 | 0.5703125 | 0.5703125
> 12 | 0.6777735948562622 | 0.7969313859939575 | 0.5703125 | 0.5703125
> 13 | 0.6750434637069702 | 0.7957834005355835 | 0.578125 | 0.5703125
> 14 | 0.6723586916923523 | 0.794644832611084 | 0.58203125 | 0.5703125
> 15 | 0.6697813272476196 | 0.7935166954994202 | 0.58984375 | 0.5703125
> 16 | 0.6672301292419434 | 0.7924058437347412 | 0.59765625 | 0.5703125
> 17 | 0.664716362953186 | 0.7913106083869934 | 0.60546875 | 0.5703125
> 18 | 0.6622849106788635 | 0.7902208566665649 | 0.6171875 | 0.5703125
> 19 | 0.6598776578903198 | 0.7891466617584229 | 0.6328125 | 0.5703125
> 20 | 0.6574945449829102 | 0.7880877256393433 | 0.640625 | 0.5703125
> 21 | 0.6551368236541748 | 0.7870438098907471 | 0.6484375 | 0.5703125
> 22 | 0.6529

### Zhang for Eq Opp

In [12]:
fairdef = 'EqOpp'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOpp', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.7590546011924744 | 0.44704705476760864 | 0.5625 | 0.5703125
> 2 | 0.7214968204498291 | 0.44609981775283813 | 0.5625 | 0.5703125
> 3 | 0.7105534672737122 | 0.4453541338443756 | 0.5625 | 0.5703125
> 4 | 0.705056369304657 | 0.4445838928222656 | 0.5625 | 0.5703125
> 5 | 0.7006242275238037 | 0.44381779432296753 | 0.5625 | 0.5703125
> 6 | 0.6966177821159363 | 0.4430620074272156 | 0.5625 | 0.5703125
> 7 | 0.692939043045044 | 0.44231176376342773 | 0.5625 | 0.5703125
> 8 | 0.6895751953125 | 0.4415557384490967 | 0.5625 | 0.5703125
> 9 | 0.6862531304359436 | 0.4408072233200073 | 0.5625 | 0.5703125
> 10 | 0.682969868183136 | 0.4400663673877716 | 0.56640625 | 0.5703125
> 11 | 0.6799836158752441 | 0.4393339157104492 | 0.5703125 | 0.5703125
> 12 | 0.6770346164703369 | 0.43860912322998047 | 0.5703125 | 0.5703125
> 13 | 0.6742199063301086 | 0.43788886070251465 | 0.578125 | 0.5703125


2026-01-05 15:29:52.001919: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 14 | 0.6714718341827393 | 0.4371683597564697 | 0.58203125 | 0.5703125
> 15 | 0.6688093543052673 | 0.43644559383392334 | 0.58984375 | 0.5703125
> 16 | 0.6661735773086548 | 0.4357300400733948 | 0.59375 | 0.5703125
> 17 | 0.6635768413543701 | 0.43501967191696167 | 0.609375 | 0.5703125
> 18 | 0.6610549688339233 | 0.4343065321445465 | 0.62109375 | 0.5703125
> 19 | 0.6585612893104553 | 0.4335971176624298 | 0.62890625 | 0.5703125
> 20 | 0.6560914516448975 | 0.4328920841217041 | 0.6328125 | 0.5703125
> 21 | 0.6536455154418945 | 0.432191401720047 | 0.6484375 | 0.5703125
> 22 | 0.651226282119751 | 0.4314950704574585 | 0.66015625 | 0.5703125
> 23 | 0.6490306854248047 | 0.43079447746276855 | 0.66015625 | 0.5703125
> 24 | 0.646888792514801 | 0.43009525537490845 | 0.6640625 | 0.5703125
> 25 | 0.6447672843933105 | 0.4293995797634125 | 0.67578125 | 0.5703125
> 26 | 0.6426652669906616 | 0.42870765924453735 | 0.67578125 | 0.5703125
> 27 | 0.6405826210975647 | 0.4280204474925995 | 0.6875 | 0.5703125
> 

## Saving into DF then CSV

In [13]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,dp,deqodds,deqopp,trade_dp,trade_deqodds,trade_deqopp,TN_a0,FP_a0,FN_a0,TP_a0,TN_a1,FP_a1,FN_a1,TP_a1
0,Zhang4DP,13,0.593750,0.794686,0.756798,0.805263,0.679678,0.665432,0.683518,9.0,18.0,2.0,17.0,5.0,3.0,3.0,7.0
1,Zhang4DP,29,0.781250,0.892857,0.926768,0.909091,0.833333,0.847810,0.840336,7.0,7.0,2.0,20.0,4.0,5.0,0.0,19.0
2,Zhang4DP,42,0.609375,0.908333,0.874684,0.902778,0.729410,0.718315,0.727612,9.0,13.0,4.0,14.0,9.0,7.0,1.0,7.0
3,Zhang4DP,55,0.687500,0.814778,0.792122,0.923611,0.745747,0.736112,0.788254,6.0,11.0,2.0,16.0,9.0,4.0,3.0,13.0
4,Zhang4DP,73,0.703125,0.788304,0.780627,0.894587,0.743282,0.739852,0.787384,12.0,6.0,7.0,20.0,2.0,4.0,2.0,11.0
5,Zhang4EqOdds,13,0.609375,0.816425,0.775317,0.805263,0.697866,0.682403,0.693757,10.0,17.0,2.0,17.0,5.0,3.0,3.0,7.0
6,Zhang4EqOdds,29,0.781250,0.892857,0.926768,0.909091,0.833333,0.847810,0.840336,7.0,7.0,2.0,20.0,4.0,5.0,0.0,19.0
7,Zhang4EqOdds,42,0.609375,0.908333,0.874684,0.902778,0.729410,0.718315,0.727612,9.0,13.0,4.0,14.0,9.0,7.0,1.0,7.0
8,Zhang4EqOdds,55,0.687500,0.814778,0.792122,0.923611,0.745747,0.736112,0.788254,6.0,11.0,2.0,16.0,9.0,4.0,3.0,13.0
9,Zhang4EqOdds,73,0.703125,0.788304,0.780627,0.894587,0.743282,0.739852,0.787384,12.0,6.0,7.0,20.0,2.0,4.0,2.0,11.0


In [14]:
result_df.to_csv(f'{data_name}-result/zhang-{epochs}.csv')